In [ ]:
from datasets import load_dataset
import re
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

In [ ]:
repo_id = "astroza/chilean-humor-raw-transcripts"
ds = load_dataset(repo_id, "segments", split="train")
segments = ds["text"]

ONLY_NUMERIC_RE = re.compile(r"^[\s0-9,.;:/+\-()\[\]{}]+$")
LETTER_RE = re.compile(r"[A-Za-zÁÉÍÓÚáéíóúÑñÜü]")

def is_numeric_noise(text: str) -> bool:
    if not text:
        return False
    t = text.strip()
    if not t:
        return False
    if LETTER_RE.search(t):
        return False
    return bool(ONLY_NUMERIC_RE.fullmatch(t))

new_ds = ds.filter(lambda x: not is_numeric_noise(x["text"]))
segments = new_ds["text"]

print(f"Total: {len(segments)} segments")

In [ ]:

spanish_stopwords = stopwords.words("spanish")

vectorizer_model = CountVectorizer(stop_words=spanish_stopwords,
                                   min_df=2,
                                   ngram_range=(1, 2),
                                   token_pattern = r"(?ui)(?:^|[^\wáéíóúñü])([a-záéíóúñü][\wáéíóúñü]+)")

In [ ]:
topic_model = BERTopic(language="multilingual",
                       vectorizer_model=vectorizer_model,
                       calculate_probabilities=True,
                       verbose=True)

topics, probs = topic_model.fit_transform(segments)

In [ ]:
freq = topic_model.get_topic_info(); freq.head(20)

In [ ]:
fig = topic_model.visualize_topics(); fig

In [ ]:
dates = new_ds["date"]

topics_over_time = topic_model.topics_over_time(docs=segments,
                                                timestamps=dates,
                                                global_tuning=True,
                                                evolution_tuning=True,
                                                nr_bins=20)

In [ ]:
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=20)

In [ ]:
dates = new_ds["date"]


decades = [(int(d[:4]) // 10) * 10 for d in dates]

topics_over_time = topic_model.topics_over_time(
    docs=segments,
    timestamps=decades,
    global_tuning=True,
    evolution_tuning=True,
)

In [ ]:
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=20)

In [ ]:
topic_model.visualize_topics_over_time(topics_over_time, topics=[11, 12, 13, 18])